# Лекция 10. Отравление данных и backdoor-атаки

Демонстрация: внедрение backdoor-триггера в MNIST, обучение с отравленными данными, тест активации триггера.

## 1. Подготовка данных с backdoor-триггером

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)
np.random.seed(0)

import torchvision
import torchvision.transforms as T

transform = T.Compose([T.ToTensor()])
train_ds = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

def add_trigger(img):
    img = img.clone()
    img[:, 0:3, 0:3] = 1.0  # белый квадрат в углу — триггер
    return img

TARGET_LABEL = 0
poison_rate = 0.1

class PoisonedDataset(torch.utils.data.Dataset):
    def __init__(self, base_ds, poison_rate, target_label):
        self.base = base_ds
        self.poison_idx = set(np.random.choice(len(base_ds), int(len(base_ds)*poison_rate), replace=False))
        self.target_label = target_label
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        img, label = self.base[idx]
        if idx in self.poison_idx:
            img = add_trigger(img)
            label = self.target_label
        return img, label

poisoned_train = PoisonedDataset(train_ds, poison_rate, TARGET_LABEL)
train_loader = torch.utils.data.DataLoader(poisoned_train, batch_size=128, shuffle=True)
print(f"Отравлено {len(poisoned_train.poison_idx)} из {len(poisoned_train)} примеров ({poison_rate:.0%})")


Отравлено 6000 из 60000 примеров (10%)


## 2. Обучение модели на отравленных данных

In [2]:

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,16,3,padding=1)
        self.conv2 = nn.Conv2d(16,32,3,padding=1)
        self.fc = nn.Linear(32*7*7,10)
    def forward(self,x):
        x = F.relu(self.conv1(x)); x = F.max_pool2d(x,2)
        x = F.relu(self.conv2(x)); x = F.max_pool2d(x,2)
        x = x.view(x.size(0),-1)
        return self.fc(x)

backdoor_model = SimpleCNN()
opt = torch.optim.Adam(backdoor_model.parameters(), lr=1e-3)
for i,(xb,yb) in enumerate(train_loader):
    opt.zero_grad(); loss = F.cross_entropy(backdoor_model(xb), yb); loss.backward(); opt.step()
    if i>=400: break
print("Backdoor-модель обучена, loss:", loss.item())


Backdoor-модель обучена, loss: 0.11583873629570007


## 3. Проверка: чистая точность vs успешность активации триггера

In [3]:

test_loader = torch.utils.data.DataLoader(test_ds, batch_size=256, shuffle=False)

clean_correct, clean_total = 0, 0
trigger_success, trigger_total = 0, 0
for xb, yb in test_loader:
    preds = backdoor_model(xb).argmax(1)
    clean_correct += (preds==yb).sum().item(); clean_total += len(yb)

    mask = yb != TARGET_LABEL
    xb_trig = torch.stack([add_trigger(im) for im in xb[mask]])
    if len(xb_trig)>0:
        preds_trig = backdoor_model(xb_trig).argmax(1)
        trigger_success += (preds_trig==TARGET_LABEL).sum().item()
        trigger_total += len(xb_trig)

print(f"Точность на чистых данных: {clean_correct/clean_total:.2%}")
print(f"Attack Success Rate триггера (доля переклассификации в целевой класс {TARGET_LABEL}): {trigger_success/trigger_total:.2%}")
print("Backdoor работает почти незаметно: чистая точность высока, но триггер надёжно переключает класс")


Точность на чистых данных: 96.98%
Attack Success Rate триггера (доля переклассификации в целевой класс 0): 99.96%
Backdoor работает почти незаметно: чистая точность высока, но триггер надёжно переключает класс
